<a href="https://colab.research.google.com/github/Schiwo/Sharing/blob/main/2026_workshop_metascience/METASCIENCE_workshop_vienna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Exploring replication rates**

In this task, you will use a simple simulation approach to explore how different research conditions influence expected replication rates.

The simulation is deterministic. This means that the same parameter values always produce the same results.


### Content

First, you will find 3 sections with functions. Function 1 simulates replication rates based on parameters such as test-power, publication-bias etc. Functions 2 and 3 can be used to plot the results returned by Function 1 to explore the influence of certain parameter on replication rates. Each Function is explained in detail.
Below, you will find 3 tasks to explore replication rates. <br><br>
**Start with "Preparation"!**


---

## Function 1: `simulate_replication`

### What this function does

Simulates a number of N primary studies. Each primary study then is matched with a replication study.

The simulation estimates how many primary study and replication study results are:

- *true positives*: A true positive occurs when a real effect exists and a study finds the effect.
- *false positives*: A false positive occurs when no real effect exists, but a study finds an effect.
- *false negatives*:  A false negative occurs when a real effect exists, but the primary study does not find the effect.
- *true negatives*: A true negative occurs when no real effect exists and the primary study does not find an effect.

Based on the result of the primary study and the replication study, expected replication rates can be calculated.

### Required parameters

The  function takes eight parameter that specify the simulation (see below).
Howver, all parameters have default values, so you don't have to select a value for each parameter.
  
### Parameter meanings

- `B`: Base rate of real effects in the research area, from 0 to 1 (default B = 0.50).
- `N`: Number of primary studies (default N = 1000).
- `aC_prim`: Critical alpha value used for significance testing in primary studies, from 0 to 1 (default aC_prim = 0.05).
- `aInf`: Extra false-positive chance from individual or systemic factors in primary studies, from 0 to 1 (default aInf = 0.20).
- `b_prim`: Primary-study test-power, meaning the chance to detect a true effect, from 0 to 1 (default b_prim = 0.80).
- `pubB`: Publication bias level, meaning the share of non-significant studies not published, from 0 to 1 (default pubB = 0.80).
- `aC_rep`: Replication-study critical alpha value used for significance testing in primary studies, from 0 to 1 (default aC_rep = 0.05).
- `b_rep`: Replication-study test-power, from 0 to 1 (default b_rep = 0.80).

### Returned data frame

The function returns a data frame with one row for each parameter combination that was simulated.

Each row describes:

1. the parameter values used for this simulation run,
2. the expected outcomes of the primary studies, and
3. the expected replication rates.

---

#### Data frame codebook

| Variable | Meaning |
|---|---|
|**Input parameters**|
| `B` | Base rate of true effects. This is the proportion of all tested hypotheses where a real effect actually exists. For example, `B = 0.30` means that 30% of the tested hypotheses are true effects. |
| `N` | Number of primary studies simulated. |
| `aC_prim` | Conventional alpha level in the primary studies. This is the probability that a primary study produces a false positive result due to chance when no real effect exists. |
| `aInf` | Additional false-positive probability caused by alpha inflation, for example due to researcher flexibility, multiple testing, selective analysis choices, or inference error. |
| `alpha_prim` | Total false-positive probability in the primary studies. This is calculated as `aC_prim + aInf`. |
| `b_prim` | Statistical power of the primary studies. This is the probability that a primary study detects a real effect when a real effect exists. |
| `pubB` | Publication bias against non-significant results. This is the proportion of non-significant primary-study results that are not published or not included in the replication sample. |
| `aC_rep` | Alpha level in the replication studies. This is the probability that a replication study produces a false positive result when no real effect exists. |
| `b_rep` | Statistical power of the replication studies. This is the probability that a replication study detects a real effect when a real effect exists. |
|**Primary-study performance**|
| `primary_pct_TP` | Percentage of all primary studies that are true positives.|
| `primary_pct_FP` | Percentage of all primary studies that are false positives.|
| `primary_pct_FN` | Percentage of all primary studies that are false negatives.|
| `primary_pct_TN` | Percentage of all primary studies that are true negatives.|
|**Replication summary**|
| `replication_N` | Number of primary studies that are followed by a replication study. This can be smaller than `N` if publication bias removes some non-significant primary-study results. |
| `replication_overall_rate_pct` | Overall percentage of replication studies that show the same result category as the corresponding primary study. For example, a primary true positive followed by a replication true positive counts as replicated; a primary false positive followed by a replication true negative does not. |
|**Replication rates by primary-study outcome**|
| `replication_rate_primary_TP_pct` | Among primary studies that were true positives, the percentage whose replication study also produces a true positive. |
| `replication_rate_primary_FP_pct` | Among primary studies that were false positives, the percentage whose replication study also produces a false positive. |
| `replication_rate_primary_FN_pct` | Among primary studies that were false negatives, the percentage whose replication study also produces a false negative. |
| `replication_rate_primary_TN_pct` | Among primary studies that were true negatives, the percentage whose replication study also produces a true negative. |

---

## Interpreting the replication-rate variables

A high value means that the replication study often gives the same type of result as the primary study.

A low value means that the replication study often gives a different type of result than the primary study.

For example:

- A high `replication_overall_rate_pct` means that many findings of the primary studies can be confirmed by replication studies (independent of the primary studies result).
- A high `replication_rate_primary_TP_pct` means that true positive findings are often confirmed in replication studies.
- A low `replication_rate_primary_FP_pct` means that many false positive findings disappear in replication studies.


### Example

Use defaults:

```r
results <- simulate_replication()
```
Compare several base-rate values at once:
```r
results <- simulate_replication(B = c(0.1, 0.3, 0.5), b_prim = 0.8)
```

In [ ]:
simulate_replication <- function(
  B = 0.50,
  N = 1000,
  aC_prim = 0.05,
  aInf = 0.20,
  b_prim = 0.80,
  pubB = 0.80,
  aC_rep = 0.05,
  b_rep = 0.80
) {
  # Collect parameters so we can support vectorized condition grids.
  params <- list(
    B = B,
    N = N,
    aC_prim = aC_prim,
    aInf = aInf,
    b_prim = b_prim,
    pubB = pubB,
    aC_rep = aC_rep,
    b_rep = b_rep
  )

  # Basic numeric validation.
  for (nm in names(params)) {
    x <- params[[nm]]
    if (!is.numeric(x) || length(x) < 1 || any(!is.finite(x))) {
      stop(sprintf("'%s' must be a numeric vector with finite values.", nm), call. = FALSE)
    }
  }

  # Any parameter can be scalar, but at most two can be vectorized (>1 value).
  vec_lengths <- vapply(params, length, integer(1))
  vectorized_params <- names(vec_lengths)[vec_lengths > 1]

  if (length(vectorized_params) > 2) {
    stop(
      sprintf(
        paste0(
          "At most two parameters may have length > 1. ",
          "You provided %d vectorized parameters: %s"
        ),
        length(vectorized_params),
        paste(vectorized_params, collapse = ", ")
      ),
      call. = FALSE
    )
  }

  if (any(vec_lengths > 8)) {
    too_long <- names(vec_lengths)[vec_lengths > 8]
    stop(
      sprintf(
        paste0(
          "Each vectorized parameter may have at most 8 elements. ",
          "These exceed 8: %s"
        ),
        paste(too_long, collapse = ", ")
      ),
      call. = FALSE
    )
  }

  # Probability parameters must stay in [0, 1].
  prob_names <- c("B", "aC_prim", "aInf", "b_prim", "pubB", "aC_rep", "b_rep")
  for (nm in prob_names) {
    x <- params[[nm]]
    if (any(x < 0 | x > 1)) {
      stop(sprintf("'%s' must be between 0 and 1.", nm), call. = FALSE)
    }
  }

  # N should be non-negative (0 is allowed, which will yield zero expected counts).
  if (any(params$N < 0)) {
    stop("'N' must be >= 0.", call. = FALSE)
  }

  # Build one row per parameter combination.
  grid <- expand.grid(params, KEEP.OUT.ATTRS = FALSE, stringsAsFactors = FALSE)

  # Effective alpha values for primary and replication studies.
  grid$alpha_prim <- grid$aC_prim + grid$aInf
  alpha_rep <- grid$aC_rep

  if (any(grid$alpha_prim > 1)) {
    stop("'aC_prim + aInf' must be <= 1 for all parameter combinations.", call. = FALSE)
  }

  # Primary-study expected counts by confusion-matrix category.
  true_effects <- grid$N * grid$B
  null_effects <- grid$N * (1 - grid$B)

  TP_prim <- true_effects * grid$b_prim
  FN_prim <- true_effects * (1 - grid$b_prim)
  FP_prim <- null_effects * grid$alpha_prim
  TN_prim <- null_effects * (1 - grid$alpha_prim)

  # Primary percentages relative to all primary studies N.
  safe_div <- function(num, den) {
    ifelse(den == 0, NA_real_, num / den)
  }

  pct_TP_prim <- safe_div(TP_prim, grid$N) * 100
  pct_FP_prim <- safe_div(FP_prim, grid$N) * 100
  pct_FN_prim <- safe_div(FN_prim, grid$N) * 100
  pct_TN_prim <- safe_div(TN_prim, grid$N) * 100

  # Publication bias: all significant findings are published, only a share of
  # non-significant findings are published.
  TP_pub <- TP_prim
  FP_pub <- FP_prim
  FN_pub <- FN_prim * (1 - grid$pubB)
  TN_pub <- TN_prim * (1 - grid$pubB)

  N_rep <- TP_pub + FP_pub + FN_pub + TN_pub

  # Replication outcomes depend on the underlying true/null status carried from
  # the corresponding primary study.
  rep_TP_from_primary_TP <- TP_pub * grid$b_rep
  rep_FN_from_primary_TP <- TP_pub * (1 - grid$b_rep)

  rep_TP_from_primary_FN <- FN_pub * grid$b_rep
  rep_FN_from_primary_FN <- FN_pub * (1 - grid$b_rep)

  rep_FP_from_primary_FP <- FP_pub * alpha_rep
  rep_TN_from_primary_FP <- FP_pub * (1 - alpha_rep)

  rep_FP_from_primary_TN <- TN_pub * alpha_rep
  rep_TN_from_primary_TN <- TN_pub * (1 - alpha_rep)

  # Cell-wise replication match rates (same cell in primary and replication).
  match_primary_TP <- safe_div(rep_TP_from_primary_TP, TP_pub)
  match_primary_FP <- safe_div(rep_FP_from_primary_FP, FP_pub)
  match_primary_FN <- safe_div(rep_FN_from_primary_FN, FN_pub)
  match_primary_TN <- safe_div(rep_TN_from_primary_TN, TN_pub)

  overall_replication_rate <- safe_div(
    rep_TP_from_primary_TP +
      rep_FP_from_primary_FP +
      rep_FN_from_primary_FN +
      rep_TN_from_primary_TN,
    N_rep
  )

  out <- data.frame(
    B = grid$B,
    N = grid$N,
    aC_prim = grid$aC_prim,
    aInf = grid$aInf,
    alpha_prim = grid$alpha_prim,
    b_prim = grid$b_prim,
    pubB = grid$pubB,
    aC_rep = grid$aC_rep,
    b_rep = grid$b_rep,
    primary_pct_TP = pct_TP_prim,
    primary_pct_FP = pct_FP_prim,
    primary_pct_FN = pct_FN_prim,
    primary_pct_TN = pct_TN_prim,
    replication_N = N_rep,
    replication_overall_rate_pct = overall_replication_rate * 100,
    replication_rate_primary_TP_pct = match_primary_TP * 100,
    replication_rate_primary_FP_pct = match_primary_FP * 100,
    replication_rate_primary_FN_pct = match_primary_FN * 100,
    replication_rate_primary_TN_pct = match_primary_TN * 100,
    primary_N_TP = TP_prim,
    primary_N_FP = FP_prim,
    primary_N_FN = FN_prim,
    primary_N_TN = TN_prim,
    published_N_TP = TP_pub,
    published_N_FP = FP_pub,
    published_N_FN = FN_pub,
    published_N_TN = TN_pub,
    stringsAsFactors = FALSE
  )

  out
}

## Function 2: `plot_replication_rate`

### What this function does

Draws a line chart of the overall replication rate (%) in dependency of a varying parameter from the output of `simulate_replication()`.

If you have simulated two varying parameters, each line shows one value of the second varying parameter.

### Required parameters

- `results`: A data frame created by `simulate_replication()`.

### Optional parameter

- `switch`: `TRUE` or `FALSE`. If two parameters vary, `TRUE` swaps which one is on the x-axis and which one is used for line style/color.

### Example

One varying parameter:

```r
r1 <- simulate_replication(B = c(0.1, 0.3, 0.5))
plot_replication_rate(r1)
```

Two varying parameters, swapped axes:

```r
r2 <- simulate_replication(B = c(0.1, 0.3, 0.5), aC_prim = c(0.05, 0.15))
plot_replication_rate(r2, switch = TRUE)
```

In [ ]:
plot_replication_rate <- function(results, switch = FALSE) {
  if (!requireNamespace("ggplot2", quietly = TRUE)) {
    stop("Package 'ggplot2' is required for plotting. Please install it first.", call. = FALSE)
  }

  if (!is.data.frame(results)) {
    stop("'results' must be a data.frame returned by simulate_replication().", call. = FALSE)
  }

  if (!is.logical(switch) || length(switch) != 1L || is.na(switch)) {
    stop("'switch' must be a single TRUE or FALSE value.", call. = FALSE)
  }

  input_parameter_columns <- c("B", "N", "aC_prim", "aInf", "b_prim", "pubB", "aC_rep", "b_rep")
  required_columns <- c(input_parameter_columns, "alpha_prim", "replication_overall_rate_pct")
  missing_columns <- setdiff(required_columns, names(results))

  if (length(missing_columns) > 0) {
    stop(
      sprintf(
        "'results' is missing required columns: %s",
        paste(missing_columns, collapse = ", ")
      ),
      call. = FALSE
    )
  }

  varying_parameters <- input_parameter_columns[
    vapply(results[input_parameter_columns], function(x) length(unique(x)) > 1, logical(1))
  ]
  fixed_parameters <- input_parameter_columns[
    vapply(results[input_parameter_columns], function(x) length(unique(x)) == 1, logical(1))
  ]

  format_value <- function(x) {
    if (is.numeric(x)) {
      formatted <- formatC(round(x, 2), format = "f", digits = 2)
      return(sub("\\.?0+$", "", formatted))
    }
    as.character(x)
  }

  fixed_caption <- if (length(fixed_parameters) == 0) {
    "Fixed parameters: none"
  } else {
    paste(
      "Fixed parameters:",
      paste(
        paste0(fixed_parameters, "=", vapply(fixed_parameters, function(nm) format_value(results[[nm]][1]), character(1))),
        collapse = " | "
      )
    )
  }

  if (length(varying_parameters) == 0) {
    stop(
      paste0(
        "The results data frame does not contain any vectorized parameter combinations. ",
        "Create results with at least one parameter passed as a vector."
      ),
      call. = FALSE
    )
  }

  if (isTRUE(switch) && length(varying_parameters) == 1) {
    stop(
      paste0(
        "'switch = TRUE' requires two vectorized parameters so the x-axis and ",
        "linetype mapping can be swapped."
      ),
      call. = FALSE
    )
  }

  x_parameter <- varying_parameters[1]
  line_parameter <- NULL

  if (length(varying_parameters) >= 2) {
    line_parameter <- varying_parameters[2]
  }

  if (isTRUE(switch)) {
    x_parameter <- varying_parameters[2]
    line_parameter <- varying_parameters[1]
  }

  plot_data <- results[order(results[[x_parameter]]), , drop = FALSE]

  # One vectorized parameter gives a single line across the x-axis values.
  if (is.null(line_parameter)) {
    return(
      ggplot2::ggplot(
        plot_data,
        ggplot2::aes(x = .data[[x_parameter]], y = .data[["replication_overall_rate_pct"]])
      ) +
        ggplot2::geom_line(linewidth = 0.8) +
        ggplot2::geom_point(size = 2) +
        ggplot2::labs(
          x = x_parameter,
          y = "Replication rate (%)",
          title = NULL,
          subtitle = NULL,
          caption = fixed_caption
        ) +
        ggplot2::scale_y_continuous(
          breaks = function(x) {
            seq(
              floor(min(x, na.rm = TRUE) / 5) * 5,
              ceiling(max(x, na.rm = TRUE) / 5) * 5,
              by = 5
            )
          }
        ) +
        ggplot2::theme_minimal()
    )
  }

  plot_data[[line_parameter]] <- as.factor(plot_data[[line_parameter]])

  ggplot2::ggplot(
    plot_data,
    ggplot2::aes(
      x = .data[[x_parameter]],
      y = .data[["replication_overall_rate_pct"]],
      color = .data[[line_parameter]],
      linetype = .data[[line_parameter]],
      group = .data[[line_parameter]]
    )
  ) +
    ggplot2::geom_line(linewidth = 0.8) +
    ggplot2::geom_point(size = 2) +
    ggplot2::labs(
      x = x_parameter,
      y = "Replication rate (%)",
      color = line_parameter,
      linetype = line_parameter,
      title = NULL,
      subtitle = NULL,
      caption = fixed_caption
    ) +
    ggplot2::scale_y_continuous(
      breaks = function(x) {
        seq(
          floor(min(x, na.rm = TRUE) / 5) * 5,
          ceiling(max(x, na.rm = TRUE) / 5) * 5,
          by = 5
        )
      }
    ) +
    ggplot2::theme_minimal()
}

## Function 3: `plot_replication_splits`

### What this function does

Draws a stacked bar chart of the replication rates (%) per primary study outcome from the output of `simulate_replication()`.

If two parameters vary, a grid is plotted.

### Required parameters

- `results`: A data frame created by `simulate_replication()`.

### Optional parameter

- `switch`: `TRUE` or `FALSE`. If two parameters vary, `TRUE` swaps which one defines columns and which one rows in the grid plot.

### Example

One varying parameter:

```r
r1 <- simulate_replication(B = c(0.1, 0.3, 0.5))
plot_replication_splits(r1)
```

Two varying parameters, swapped axes:

```r
r2 <- simulate_replication(B = c(0.1, 0.3, 0.5), aC_prim = c(0.05, 0.15))
plot_replication_splits(r2, switch = TRUE)
```


In [ ]:
plot_replication_splits <- function(results, switch = FALSE) {
  if (!requireNamespace("ggplot2", quietly = TRUE)) {
    stop("Package 'ggplot2' is required for plotting. Please install it first.", call. = FALSE)
  }

  if (!is.data.frame(results)) {
    stop("'results' must be a data.frame returned by simulate_replication().", call. = FALSE)
  }

  if (!is.logical(switch) || length(switch) != 1L || is.na(switch)) {
    stop("'switch' must be a single TRUE or FALSE value.", call. = FALSE)
  }

  input_parameter_columns <- c("B", "N", "aC_prim", "aInf", "b_prim", "pubB", "aC_rep", "b_rep")
  required_columns <- c(
    input_parameter_columns,
    "alpha_prim",
    "primary_pct_TP", "primary_pct_FP", "primary_pct_FN", "primary_pct_TN",
    "replication_rate_primary_TP_pct", "replication_rate_primary_FP_pct",
    "replication_rate_primary_FN_pct", "replication_rate_primary_TN_pct"
  )
  missing_columns <- setdiff(required_columns, names(results))

  if (length(missing_columns) > 0) {
    stop(
      sprintf(
        "'results' is missing required columns: %s",
        paste(missing_columns, collapse = ", ")
      ),
      call. = FALSE
    )
  }

  varying_parameters <- input_parameter_columns[
    vapply(results[input_parameter_columns], function(x) length(unique(x)) > 1, logical(1))
  ]
  fixed_parameters <- input_parameter_columns[
    vapply(results[input_parameter_columns], function(x) length(unique(x)) == 1, logical(1))
  ]

  format_value <- function(x) {
    if (is.numeric(x)) {
      formatted <- formatC(round(x, 2), format = "f", digits = 2)
      return(sub("\\.?0+$", "", formatted))
    }
    as.character(x)
  }

  fixed_caption <- if (length(fixed_parameters) == 0) {
    "Fixed parameters: none"
  } else {
    paste(
      "Fixed parameters:",
      paste(
        paste0(fixed_parameters, "=", vapply(fixed_parameters, function(nm) format_value(results[[nm]][1]), character(1))),
        collapse = " | "
      )
    )
  }

  if (length(varying_parameters) > 2) {
    stop(
      sprintf(
        paste0(
          "'results' contains more than two varying input parameters (%s). ",
          "Please provide output where at most two input parameters vary."
        ),
        paste(varying_parameters, collapse = ", ")
      ),
      call. = FALSE
    )
  }

  one_param <- NULL
  row_param <- NULL
  col_param <- NULL

  if (length(varying_parameters) == 1) {
    one_param <- varying_parameters[1]
  }

  if (length(varying_parameters) == 2) {
    row_param <- varying_parameters[1]
    col_param <- varying_parameters[2]
    if (isTRUE(switch)) {
      row_param <- varying_parameters[2]
      col_param <- varying_parameters[1]
    }
  }

  primary_cols <- c(TP = "primary_pct_TP", FP = "primary_pct_FP", FN = "primary_pct_FN", TN = "primary_pct_TN")
  match_cols <- c(
    TP = "replication_rate_primary_TP_pct",
    FP = "replication_rate_primary_FP_pct",
    FN = "replication_rate_primary_FN_pct",
    TN = "replication_rate_primary_TN_pct"
  )

  palette <- c(
    TP_primary = "#dcf6dc",
    FP_primary = "#ffdada",
    FN_primary = "#f3def1",
    TN_primary = "#d6efec",
    TP_rep = "#0b5d1e",
    FP_rep = "#7f1212",
    FN_rep = "#5f1b48",
    TN_rep = "#0f4f46"
  )

  gap <- 0
  bar_width <- 0.5
  seg_list <- vector("list", nrow(results))
  rep_list <- vector("list", nrow(results))
  txt_list <- vector("list", nrow(results))
  bar_txt_list <- vector("list", nrow(results))

  for (i in seq_len(nrow(results))) {
    row_i <- results[i, , drop = FALSE]

    counts <- c(
      TP = as.numeric(row_i[[primary_cols["TP"]]]),
      FP = as.numeric(row_i[[primary_cols["FP"]]]),
      FN = as.numeric(row_i[[primary_cols["FN"]]]),
      TN = as.numeric(row_i[[primary_cols["TN"]]])
    )

    match_rates <- c(
      TP = as.numeric(row_i[[match_cols["TP"]]]),
      FP = as.numeric(row_i[[match_cols["FP"]]]),
      FN = as.numeric(row_i[[match_cols["FN"]]]),
      TN = as.numeric(row_i[[match_cols["TN"]]])
    )
    match_rates[is.na(match_rates)] <- 0
    match_rates <- pmax(pmin(match_rates, 100), 0)

    replicated <- counts * match_rates / 100

    x_pos_min <- 0
    x_pos_max <- bar_width
    x_neg_min <- x_pos_max + gap
    x_neg_max <- x_neg_min + bar_width

    make_stack <- function(cells, x_min, x_max) {
      y_cursor <- 0
      out <- data.frame(stringsAsFactors = FALSE)

      for (cell in cells) {
        # Use absolute percentages so y-axis directly indicates % of all studies.
        h <- counts[cell] / 100
        y_min <- y_cursor
        y_max <- y_cursor + h
        rep_h <- replicated[cell] / 100

        out <- rbind(
          out,
          data.frame(
            cell = cell,
            xmin = x_min,
            xmax = x_max,
            ymin = y_min,
            ymax = y_max,
            rymax = y_min + rep_h,
            primary_pct = counts[cell],
            replicated_pct = replicated[cell],
            stringsAsFactors = FALSE
          )
        )
        y_cursor <- y_max
      }

      out
    }

    left <- make_stack(c("FP", "TP"), x_pos_min, x_pos_max)
    right <- make_stack(c("TN", "FN"), x_neg_min, x_neg_max)
    stacks <- rbind(left, right)

    for (nm in input_parameter_columns) {
      stacks[[nm]] <- row_i[[nm]]
    }

    seg_df <- stacks
    seg_df$fill_key <- paste0(seg_df$cell, "_primary")

    rep_df <- stacks
    rep_df$ymax <- pmin(rep_df$rymax, rep_df$ymax)
    rep_df$fill_key <- paste0(rep_df$cell, "_rep")
    rep_df <- rep_df[rep_df$ymax > rep_df$ymin, , drop = FALSE]

    txt_df <- stacks
    txt_df$xm <- (txt_df$xmin + txt_df$xmax) / 2
    txt_df$ym <- (txt_df$ymin + txt_df$ymax) / 2
    txt_df$h <- txt_df$ymax - txt_df$ymin
    txt_df$label <- sprintf(
      "%s: %.1f%%\n%s replicated: %.1f%%",
      txt_df$cell,
      txt_df$primary_pct,
      txt_df$cell,
      txt_df$replicated_pct
    )
    txt_df <- txt_df[txt_df$h >= 0.06, , drop = FALSE]

    bar_df <- data.frame(
      x = c((x_pos_min + x_pos_max) / 2, (x_neg_min + x_neg_max) / 2),
      y = c(-0.06, -0.06),
      label = c("Primary positive", "Primary negative"),
      stringsAsFactors = FALSE
    )
    for (nm in input_parameter_columns) {
      bar_df[[nm]] <- row_i[[nm]]
    }

    seg_list[[i]] <- seg_df
    rep_list[[i]] <- rep_df
    txt_list[[i]] <- txt_df
    bar_txt_list[[i]] <- bar_df
  }

  segments <- do.call(rbind, seg_list)
  replicated_segments <- do.call(rbind, rep_list)
  labels <- do.call(rbind, txt_list)
  bar_labels <- do.call(rbind, bar_txt_list)

  p <- ggplot2::ggplot() +
    ggplot2::geom_rect(
      data = segments,
      ggplot2::aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax, fill = fill_key),
      color = "grey35",
      linewidth = 0.35
    ) +
    ggplot2::geom_rect(
      data = replicated_segments,
      ggplot2::aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax, fill = fill_key),
      color = "white",
      linewidth = 0.2
    ) +
    ggplot2::geom_text(
      data = labels,
      ggplot2::aes(x = xm, y = ym, label = label),
      size = 2.6,
      lineheight = 0.9
    ) +
    ggplot2::geom_text(
      data = bar_labels,
      ggplot2::aes(x = x, y = y, label = label),
      size = 3
    ) +
    ggplot2::scale_fill_manual(
      values = palette,
      breaks = c(
        "TP_primary", "TP_rep",
        "FP_primary", "FP_rep",
        "FN_primary", "FN_rep",
        "TN_primary", "TN_rep"
      ),
      labels = c(
        "TP not replicated", "TP replicated",
        "FP not replicated", "FP replicated",
        "FN not replicated", "FN replicated",
        "TN not replicated", "TN replicated"
      ),
      name = "Color meaning"
    ) +
    ggplot2::coord_cartesian(xlim = c(-0.02, 1.02), ylim = c(-0.10, 1.02), clip = "off") +
    ggplot2::scale_x_continuous(breaks = NULL) +
    ggplot2::scale_y_continuous(
      breaks = c(0, 0.25, 0.5, 0.75, 1),
      labels = c("0%", "25%", "50%", "75%", "100%")
    ) +
    ggplot2::labs(
      x = NULL,
      y = "% of studies",
      title = NULL,
      subtitle = NULL,
      caption = fixed_caption
    ) +
    ggplot2::theme_minimal(base_size = 11) +
    ggplot2::theme(
      panel.grid.major.x = ggplot2::element_blank(),
      panel.grid.minor = ggplot2::element_blank(),
      legend.position = "right"
    )

  if (length(varying_parameters) == 0) {
    return(p)
  }

  if (length(varying_parameters) == 1) {
    p <- p + ggplot2::facet_wrap(stats::as.formula(paste("~", one_param)), nrow = 1, labeller = ggplot2::label_both)
    return(p)
  }

  p + ggplot2::facet_grid(stats::as.formula(paste(row_param, "~", col_param)), labeller = ggplot2::label_both)
}


## Preparation

Before you start, you might want to save this notebook to your Google Drive, so that your changes are saved. Select `File` > `Save a copy in Drive`. <br>
To get started, run the cells with the functions once (so that they become available in your R environment). <br>
Next, run the simulation once with the default values that are used for the simulation if no values are provided for the parameters:

```r
results_default <- simulate_replication()
results_default
```

You will encounter an error, if you try to use the `plot_replication_rate()` function. This is because this plotting function is supposed to compare the replication rate in dependency of one or two varying parameter(s). <br>
```r
# This will throw an error
plot_replication_rate(results_default)
```
To create a data frame with varying parameters, simply pass them as a vector `xyz=c(00.12, 00.23, 00.45)`.

```r
results_varying_power <- simulate_replication(b_prim = c(.5, .7, .9))
results_varying_power_and_alphaCritical <- simulate_replication(b_prim = c(.5, .7, .9), aC_prim = c(.1, .05, .001))

plot_replication_rate(results_varying_power)
plot_replication_rate(results_varying_power_and_alphaCritical)
```

---



In [ ]:
# Your code goes in here!


## Task 1: Explore the influence of each parameter

Vary one parameter at a time while keeping all other parameters at their default values (you can vary up to two parameters but not more at the same time).

For example:

```r
r_B <- simulate_replication(B = c(0.1, 0.3, 0.5, 0.7, 0.9))
plot_replication_rate(r_B)
```

Repeat this for the following parameters:

```r
B
aC_prim
aInf
b_prim
pubB
aC_rep
b_rep
```

Which parameters have larger influence on the replication rate `replication_overall_rate_pct`?

---


In [ ]:
# Your code goes in here!

## Task 2: Find a realistic parameter set that produces about 49% replication

[Tyner, Abatayo, Daley et al. (2026)]( https://doi.org/10.1038/s41586-025-10078-y) reported a median replication success estimate of about 49.3% across different evaluation methods.

Your task is to find a realistic set of parameters that produces a replication rate close to 49%.

Start with this example and modify it:

```r
r_target <- simulate_replication(
  B = 0.3,
  N = 1000,
  aC_prim = 0.05,
  aInf = 0.10,
  b_prim = 0.50,
  pubB = 0.80,
  aC_rep = 0.05,
  b_rep = 0.80
)

r_target
```

### Your goal

Find a parameter set where the overall replication rate is approximately 49%.
```

### Questions

1. Which parameter values did you choose?
2. Why do you think these values are realistic or plausible?
3. Could very different parameter combinations also produce a similar replication rate?



In [ ]:
# Your code goes in here!

## Task 3: How can scientists influence the parameters?

For each parameter, think about what individual researchers, journals, funders, reviewers, or the scientific community could do to improve replicability.

| Parameter | Meaning |
|---|---|
| `B` | Base rate of true effects |
| `aC_prim` | Critical alpha value used for significance testing in primary studies |
| `aInf` | Extra false-positive chance from individual or systemic factors in primary studies |
| `b_prim` | Primary-study test-power, meaning the chance to detect a true effect |
| `pubB` | Publication bias level, meaning the share of non-significant studies not published |
| `aC_rep` | Replication-study critical alpha value used for significance testing in primary studies |
| `b_rep` | Replication-study test-power |

### Questions

1. Which parameters can individual researchers influence directly?
2. Which parameters require changes at the level of journals, funders, or scientific institutions?
3. Which changes would probably have the largest effect on replication rates with the least amount of "effort"?
4. In how far could the simulation be an oversimplification of the reality in science?

---


In [ ]:
# Your code goes in here (if needed)!